# TCC — estático e temporal consolidados
Comparação de CNNs congeladas para estimar largura. Fluxo: configuração → base → features → quatro análises estáticas → duas análises temporais → gráficos salvos.

O temporal usa **a mesma extração da execução atual**. K=0 é estático pareado aos mesmos frames elegíveis de K=1…100. Não é automaticamente o resultado estático com todos os frames.

**Novos resultados:** `TCC/resultados/tcc_consolidado_v1/execucoes`. Nenhum resultado do notebook anterior é apagado ou misturado automaticamente.

As correspondências físicas imagem–largura continuam provisórias; ordenação e consistência estrutural não as comprovam. Rótulos interpolados não são medições independentes por frame.

## 1. Configuração única
Escolha CNN, base e parâmetros. Plano: sementes 42/43; K=0/1/5/10/25/50/100. Os treinamentos começam desativados. Ative `EXECUTAR_ESTATICO` e/ou `EXECUTAR_TEMPORAL` somente quando quiser treinar.

**Nova execução:** `RETOMAR_EXECUCAO=''`, execute em ordem. **Retomada sem reler imagens:** informe a pasta deste template e execute esta configuração → seção 6 (funções, preparação, validação) → etapa pendente. Mantenha os mesmos hiperparâmetros. Os painéis só precisam das funções e dos arquivos no Drive.

In [ ]:
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
from datetime import datetime, timezone
from time import perf_counter
import hashlib, json, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display
from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/TCC')
ZIP_PATH = PROJECT_ROOT / 'Dados Experimentos/experimentos_taguchi_400p_cropped_images.zip'
SOFIA_PATH = PROJECT_ROOT / 'Dados Experimentos/cordon_width_profiles_dense_oriented.csv'
OUTPUT_ROOT = PROJECT_ROOT / 'resultados/tcc_consolidado_v1'

POOLING = 'max'
TAMANHO_COMUM = None

ARQUITETURA = 'DenseNet169'  # , , EfficientNetB0,InceptionResNetV2,ResNet50,DenseNet169
USAR_PARAMETROS = True

EXECUTAR_ESTATICO = False  # True autoriza os 4 protocolos estáticos.
EXECUTAR_TEMPORAL = False # True autoriza central + aleatório (2.028 fits).
##RETOMAR_EXECUCAO = '/content/drive/MyDrive/TCC/resultados/tcc_consolidado_v1/execucoes/EfficientNetB0_20261023T221736_165705Z'
RETOMAR_EXECUCAO = ''  # Vazio: nova execução; informe uma pasta para retomar.
REGERAR_FEATURES = True

USAR_DATASET_ANTIGO = False





# Plano único — não altere a configuração ao retomar uma execução.
SEMENTES = (42, 43)
SEMENTE_DIVISAO = 42
COLUNAS_TEMPORAIS = (0, 1, 5, 10, 25, 50, 100)
GAP_FRAMES = 100
EPOCAS_AUX = 50
     # Caminho exato de uma pasta NOVA deste template; vazio cria execução.
PASTA_FEATURES_SALVAS = '' # Se REGERAR_FEATURES=False e não for retomada, indique uma extração compatível.
SEMENTE = 42
BATCH_EXTRACAO = 30
BATCH_REGRESSAO = 128
EPOCAS = 300
PACIENCIA = 20
TAXA_APRENDIZADO = 0.001
EXPERIMENTOS = [f'L{i}' for i in range(1, 10)]
INTERVALOS = {
    'L1': (1450, 5713), 'L2': (600, 4550), 'L3': (350, 4200),
    'L4': (750, 3800), 'L5': (600, 3700), 'L6': (650, 3600),
    'L7': (650, 2400), 'L8': (800, 2650), 'L9': (750, 2650),
}


COLUNAS_PARAMETROS = ['current', 'nozzle_speed', 'mass_flow_rate']
# Mesmos valores registrados no notebook anterior; unidades da fonte devem ser
# mantidas no texto do TCC. Aqui são apenas os valores numéricos originais.
PARAMETROS = pd.DataFrame([
    ['L1',120,60,10], ['L2',160,60,15], ['L3',200,60,20],
    ['L4',120,80,10], ['L5',160,80,15], ['L6',200,80,20],
    ['L7',120,100,10], ['L8',160,100,15], ['L9',200,100,20],
], columns=['experimento'] + COLUNAS_PARAMETROS)
keras.utils.set_random_seed(SEMENTE)
tf.config.experimental.enable_op_determinism()
print('Arquitetura:', ARQUITETURA, '| Parâmetros como entrada:', USAR_PARAMETROS)
print('TensorFlow:', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))


## 2. Imagens e metadados ordenados

O ZIP permanece no Drive. As imagens são extraídas no disco temporário do Colab para
evitar milhares de leituras remotas. O inventário do ZIP identifica a origem do conjunto.
Um PNG já extraído é reaproveitado apenas se o tamanho esperado estiver presente.
As features serão sempre calculadas novamente na etapa 5.


In [ ]:
with ZipFile(ZIP_PATH) as arquivo:
    membros = []
    for item in arquivo.infolist():
        partes = PurePosixPath(item.filename).parts
        if (len(partes) == 3 and partes[0] == 'experimentos_taguchi_400p_cropped_images'
                and partes[1] in EXPERIMENTOS and partes[2].lower().endswith('.png')):
            membros.append(item)
    inventario = sorted((m.filename, m.file_size, m.CRC) for m in membros)
    zip_id = hashlib.sha256(json.dumps(inventario).encode()).hexdigest()[:16]
    EXTRACAO_ROOT = Path('/content/tcc_estatico_new') / zip_id
    DATASET_ROOT = EXTRACAO_ROOT / 'experimentos_taguchi_400p_cropped_images'
    for item in membros:
        destino = EXTRACAO_ROOT / item.filename
        if not destino.is_file() or destino.stat().st_size != item.file_size:
            arquivo.extract(item, EXTRACAO_ROOT)

linhas = []
for experimento in EXPERIMENTOS:
    arquivos = sorted((DATASET_ROOT / experimento).glob('*.png'), key=lambda p: int(p.stem))
    for posicao, caminho in enumerate(arquivos):
        linhas.append({'experimento': experimento, 'frame': int(caminho.stem),
                       'posicao_no_experimento': posicao,
                       'caminho_relativo': caminho.relative_to(DATASET_ROOT).as_posix()})
metadados = pd.DataFrame(linhas)
metadados.insert(0, 'indice_global', np.arange(len(metadados)))
perfil_sofia = pd.read_csv(SOFIA_PATH)
display(metadados.head())
print(f'{len(metadados):,} imagens | origem ZIP: {zip_id}')


In [ ]:
# VALIDAÇÃO 1 — imagens, ordem e perfil de entrada (não modifica rótulos)
assert len(metadados) == len(membros) > 0, 'Inventário de imagens incompleto.'
assert set(metadados.experimento) == set(EXPERIMENTOS), 'Experimento ausente.'
assert not metadados.duplicated(['experimento','frame']).any(), 'Frame duplicado.'
assert metadados.groupby('experimento').frame.apply(lambda s: s.is_monotonic_increasing).all()
assert {'experimento','posicao_temporal_normalizada','largura_mm'} <= set(perfil_sofia)
assert set(perfil_sofia.experimento) == set(EXPERIMENTOS)
for exp, perfil in perfil_sofia.groupby('experimento'):
    perfil = perfil.sort_values('posicao_temporal_normalizada')
    x = perfil.posicao_temporal_normalizada.to_numpy()
    y = perfil.largura_mm.to_numpy()
    assert len(x) > 1 and np.isfinite(x).all() and np.isfinite(y).all(), exp
    assert (np.diff(x) > 0).all() and (y > 0).all(), exp
    assert np.allclose([x[0],x[-1]], [0,1]), f'{exp}: cobertura incompleta do perfil.'
display(metadados.groupby('experimento').agg(
    imagens=('frame','size'), primeiro_frame=('frame','min'), ultimo_frame=('frame','max')))
print('Integridade estrutural conferida. Correspondência física continua como hipótese documentada.')
display(metadados.head(6))


## 3. Associar o perfil da Sofia

Selecionamos os frames pelos intervalos da configuração. A posição é calculada por
`(frame − início)/(fim − 1 − início)`, preservando possíveis lacunas da numeração.
Nos dados completos e consecutivos, isso equivale ao `linspace` usado anteriormente.
O começo e o fim selecionados são associados aos extremos do perfil: essa é uma
**hipótese de alinhamento**, não uma medição física comprovada. A orientação do CSV é preservada.


In [ ]:
def associar_perfis(metadados, perfis, intervalos):
    partes = []
    for exp, (inicio, fim) in intervalos.items():
        imagens = metadados.loc[(metadados.experimento == exp)
                               & (metadados.frame >= inicio) & (metadados.frame < fim)].copy()
        perfil = perfis.loc[perfis.experimento == exp].sort_values('posicao_temporal_normalizada')
        posicao = (imagens.frame.to_numpy() - inicio) / (fim - 1 - inicio)
        imagens['posicao_normalizada'] = posicao
        imagens['y'] = np.interp(posicao, perfil.posicao_temporal_normalizada,
                                 perfil.largura_mm).astype(np.float32)
        imagens['origem_y'] = 'sofia_perfil_denso_orientado_interpolacao_linear'
        partes.append(imagens)
    base = pd.concat(partes, ignore_index=True)
    base.insert(0, 'indice_dataset', np.arange(len(base)))
    return base

dataset = associar_perfis(metadados, perfil_sofia, INTERVALOS)
dataset = dataset.merge(PARAMETROS, on='experimento', how='left', validate='many_to_one', sort=False)
display(dataset.groupby('experimento', sort=False).head(2))


In [ ]:
# VALIDAÇÃO 2 — alinhamento estrutural da base e identificação reprodutível
assert np.array_equal(dataset.indice_dataset, np.arange(len(dataset)))
assert not dataset.duplicated(['experimento','frame']).any()
assert np.isfinite(dataset[['y'] + COLUNAS_PARAMETROS].to_numpy()).all()
assert (dataset.y > 0).all()
for exp, (inicio, fim) in INTERVALOS.items():
    frames = dataset.loc[dataset.experimento == exp, 'frame'].to_numpy()
    assert np.array_equal(frames, np.arange(inicio, fim)), f'{exp}: frames ausentes ou fora de ordem.'

sofia_sha256 = hashlib.sha256(SOFIA_PATH.read_bytes()).hexdigest()
assinatura = dataset.to_csv(index=False, float_format='%.9g').encode()
DATASET_ID = hashlib.sha256(assinatura + zip_id.encode() + sofia_sha256.encode()).hexdigest()[:16]
BASE_DIR = OUTPUT_ROOT / 'bases' / DATASET_ID
BASE_DIR.mkdir(parents=True, exist_ok=True)
dataset.to_csv(BASE_DIR / 'dataset_rotulado.csv', index=False)
metadados.to_csv(BASE_DIR / 'metadados_imagens.csv', index=False)
(BASE_DIR / 'origem.json').write_text(json.dumps({
    'dataset_id': DATASET_ID, 'zip_inventario_id': zip_id, 'sofia_sha256': sofia_sha256,
    'intervalos_frame_fim_exclusivo': INTERVALOS,
    'associacao': 'linear_por_numero_frame_entre_extremos_provisorios',
    'correspondencia_fisica_confirmada': False,
}, indent=2), encoding='utf-8')
display(dataset.groupby('experimento').agg(n=('y','size'), frame_inicial=('frame','min'),
    frame_final=('frame','max'), y_min_mm=('y','min'), y_medio_mm=('y','mean'), y_max_mm=('y','max')))
print('Base:', DATASET_ID, '|', dataset.shape, '| Salva em:', BASE_DIR)


### Visualizar rótulos e imagens

As curvas mostram o perfil fornecido e a reamostragem nos frames. Os pontos laranja
são rótulos interpolados, **não medições independentes**. A grade de imagens mostra
início, meio e fim do recorte. Esses gráficos ajudam a encontrar problemas, mas não
estabelecem sozinhos a correspondência física entre cordão e vídeo.


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for ax, exp in zip(axes.flat, EXPERIMENTOS):
    p = perfil_sofia.loc[perfil_sofia.experimento == exp].sort_values('posicao_temporal_normalizada')
    d = dataset.loc[dataset.experimento == exp]
    ax.plot(p.posicao_temporal_normalizada, p.largura_mm, label='Perfil Sofia', color='navy')
    amostra = d.iloc[np.linspace(0, len(d)-1, 35).astype(int)]
    ax.scatter(amostra.posicao_normalizada, amostra.y, s=12, color='darkorange', label='Y nos frames')
    ax.set(title=exp, xlabel='Posição normalizada', ylabel='Largura (mm)')
    ax.grid(alpha=.25)
axes.flat[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(BASE_DIR / 'perfis_e_rotulos.png', dpi=160)
plt.show()

fig, axes = plt.subplots(9, 3, figsize=(10, 22))
for linha, exp in enumerate(EXPERIMENTOS):
    d = dataset.loc[dataset.experimento == exp]
    for ax, pos in zip(axes[linha], [0, len(d)//2, len(d)-1]):
        r = d.iloc[pos]
        ax.imshow(plt.imread(DATASET_ROOT / r.caminho_relativo))
        ax.set_title(f'{exp} | frame {r.frame} | y={r.y:.2f} mm', fontsize=9)
        ax.axis('off')
fig.tight_layout()
fig.savefig(BASE_DIR / 'imagens_inicio_meio_fim.png', dpi=120)
plt.show()


In [ ]:
if USAR_DATASET_ANTIGO:
    print("Aviso: teste ativado; carregando a base antiga adaptada.")

    CAMINHO_DATASET_OLD = (
        PROJECT_ROOT / "resultados" / "inceptionresnetv2"
        / "metadados_rotulados_old_adaptado.csv"
    )
    dataset_old = pd.read_csv(CAMINHO_DATASET_OLD)
    dataset = dataset_old.rename(columns={"indice_old": "indice_dataset"}).copy()

    # A BASE_DIR da etapa 6 pertence à base da Sofia. Cria uma pasta própria
    # para a base _old, evitando comparar ou identificar bases diferentes como iguais.
    assinatura_old = dataset.to_csv(index=False, float_format="%.9g").encode()
    DATASET_ID = hashlib.sha256(
        assinatura_old + CAMINHO_DATASET_OLD.read_bytes()
    ).hexdigest()[:16]
    BASE_DIR = OUTPUT_ROOT / "bases" / f"{DATASET_ID}_old"
    BASE_DIR.mkdir(parents=True, exist_ok=True)
    dataset.to_csv(BASE_DIR / "dataset_rotulado.csv", index=False)
    (BASE_DIR / "origem.json").write_text(json.dumps({
        "dataset_id": DATASET_ID,
        "origem": str(CAMINHO_DATASET_OLD),
        "metodologia_label": "dataset_rotulado_old_adaptado",
        "linhas": len(dataset),
        "observacao": "Base antiga escolhida explicitamente para teste; não é a base rotulada da Sofia.",
    }, indent=2), encoding="utf-8")

    print(f"Base _old: {dataset.shape} | ID: {DATASET_ID} | Salva em: {BASE_DIR}")
    display(dataset.head(10))
else:
    print("Teste _old desativado. A execução continua com a base rotulada da Sofia.")


## 4. CNN configurável e leitura em lotes

Pesos ImageNet congelados; cada arquitetura usa seu próprio pré-processamento.
EfficientNetB0 recebe pixels entre 0 e 255: a normalização já integra o modelo.
`NUMERO_FEATURES_CNN` vem da saída real da CNN. A leitura paralela conserva a ordem
dos metadados. O pequeno piloto aquece a execução antes da medição completa.


In [ ]:
CATALOGO = {
    'InceptionResNetV2': (keras.applications.InceptionResNetV2, (299,299), keras.applications.inception_resnet_v2.preprocess_input),
    'ResNet50': (keras.applications.ResNet50, (224,224), keras.applications.resnet50.preprocess_input),
    'DenseNet169': (keras.applications.DenseNet169, (224,224), keras.applications.densenet.preprocess_input),
    'EfficientNetB0': (keras.applications.EfficientNetB0, (224,224), keras.applications.efficientnet.preprocess_input),
}
construtor, tamanho_padrao, preprocessar = CATALOGO[ARQUITETURA]
TAMANHO_ENTRADA = TAMANHO_COMUM or tamanho_padrao
keras.backend.clear_session()
keras.utils.set_random_seed(SEMENTE)
extrator = construtor(weights='imagenet', include_top=False, pooling=POOLING,
                     input_shape=(*TAMANHO_ENTRADA, 3))
extrator.trainable = False
NUMERO_FEATURES_CNN = int(extrator.output_shape[-1])

def criar_dataset_imagens(tabela):
    caminhos = [str(DATASET_ROOT / c) for c in tabela.caminho_relativo]
    def carregar(caminho):
        imagem = tf.image.decode_png(tf.io.read_file(caminho), channels=3)
        imagem = tf.image.resize(imagem, TAMANHO_ENTRADA, method='bilinear', antialias=True)
        return preprocessar(tf.cast(imagem, tf.float32))
    return (tf.data.Dataset.from_tensor_slices(caminhos)
            .map(carregar, num_parallel_calls=tf.data.AUTOTUNE, deterministic=True)
            .batch(BATCH_EXTRACAO).prefetch(tf.data.AUTOTUNE))

piloto = dataset.iloc[:min(60, len(dataset))]
features_piloto = extrator.predict(criar_dataset_imagens(piloto), verbose=0)
print(f'{ARQUITETURA}: {TAMANHO_ENTRADA} → {NUMERO_FEATURES_CNN} features por imagem')


In [ ]:
# VALIDAÇÃO 3 — dimensão, valores e ordem usando inferências individuais
assert features_piloto.shape == (len(piloto), NUMERO_FEATURES_CNN)
assert np.isfinite(features_piloto).all()
for i in [0, len(piloto)-1]:
    individual = extrator.predict(criar_dataset_imagens(piloto.iloc[[i]]), verbose=0)[0]
    np.testing.assert_allclose(features_piloto[i], individual, rtol=2e-3, atol=2e-4)
print('Piloto conferido: primeira e última imagem correspondem aos respectivos vetores.')


## 5. Extração completa e custo computacional

Defina `REGERAR_FEATURES = True` para recalcular todas as features ou `False` para carregar a extração indicada em `PASTA_FEATURES_SALVAS`. Ao reutilizar, conferimos a identidade da base, a ordem das imagens, a arquitetura, a resolução, o pooling e a dimensão dos vetores. As etapas anteriores de preparação e o piloto continuam necessários.

Cada execução recebe uma pasta exclusiva para preservar os resultados anteriores. A próxima célula valida e salva a matriz `.npy`, o índice e os custos. Na reutilização, tempo, vazão, batch e dispositivo mantêm os valores da extração original; a origem fica registrada.

O tempo de extração inclui leitura, redimensionamento, pré-processamento e CNN; exclui download dos pesos e gravação no Drive. `ms/imagem` é o tempo médio amortizado em lotes. Compare custo no mesmo hardware, batch, precisão e política de resolução.


In [ ]:
# True: recalcula todas as features. False: carrega a extração indicada abaixo.

if RETOMAR_EXECUCAO:
    REGERAR_FEATURES=False
    PASTA_FEATURES_SALVAS=Path(RETOMAR_EXECUCAO)
elif not REGERAR_FEATURES:
    if not PASTA_FEATURES_SALVAS: raise ValueError('Informe PASTA_FEATURES_SALVAS.')
    PASTA_FEATURES_SALVAS=Path(PASTA_FEATURES_SALVAS)

if REGERAR_FEATURES:
    inicio = perf_counter()
    features = extrator.predict(criar_dataset_imagens(dataset), verbose=1).astype(np.float32)
    tempo_extracao = perf_counter() - inicio
    gpus = tf.config.list_physical_devices('GPU')
    dispositivo = tf.config.experimental.get_device_details(gpus[0]).get('device_name', 'GPU') if gpus else 'CPU'
    custo_extrator = {
        'arquitetura': ARQUITETURA, 'entrada': str(TAMANHO_ENTRADA), 'pooling': POOLING,
        'features_cnn': NUMERO_FEATURES_CNN, 'parametros_extrator': int(extrator.count_params()),
        'tempo_extracao_s': tempo_extracao, 'imagens_s': len(dataset) / tempo_extracao,
        'ms_imagem_lote': 1000 * tempo_extracao / len(dataset),
        'memoria_features_MiB': features.nbytes / 1024**2, 'batch_extracao': BATCH_EXTRACAO,
        'dispositivo': dispositivo, 'dtype': str(features.dtype),
    }
else:
    # Shape sozinho não garante que os vetores pertencem às imagens atuais.
    necessarios = ['features.npy', 'indice_features.csv', 'custo_extrator.csv', 'configuracao.json']
    faltantes = [nome for nome in necessarios if not (PASTA_FEATURES_SALVAS / nome).is_file()]
    if faltantes:
        raise FileNotFoundError(f'Extração incompleta em {PASTA_FEATURES_SALVAS}: {faltantes}. Confira a pasta ou use REGERAR_FEATURES=True.')
    config_salva = json.loads((PASTA_FEATURES_SALVAS / 'configuracao.json').read_text())
    custo_extrator = pd.read_csv(PASTA_FEATURES_SALVAS / 'custo_extrator.csv').iloc[0].to_dict()
    if (config_salva['dataset_id'] != DATASET_ID
            or custo_extrator['arquitetura'] != ARQUITETURA
            or custo_extrator['entrada'] != str(TAMANHO_ENTRADA)
            or custo_extrator['pooling'] != POOLING
            or int(custo_extrator['features_cnn']) != NUMERO_FEATURES_CNN):
        raise ValueError('A extração salva não corresponde à base ou à CNN atual. Escolha outra pasta ou use REGERAR_FEATURES=True.')
    indice_salvo = pd.read_csv(PASTA_FEATURES_SALVAS / 'indice_features.csv')
    colunas_identidade = ['indice_dataset', 'experimento', 'frame', 'caminho_relativo']
    pd.testing.assert_frame_equal(
        indice_salvo[colunas_identidade].reset_index(drop=True),
        dataset[colunas_identidade].reset_index(drop=True), check_dtype=False,
    )
    features = np.load(PASTA_FEATURES_SALVAS / 'features.npy', mmap_mode='r', allow_pickle=False)
    if features.shape != (len(dataset), NUMERO_FEATURES_CNN) or features.dtype != np.float32:
        raise ValueError('Dimensão ou dtype das features salvas incompatível.')
    # Preserva tempo e dispositivo da extração original, não o tempo de carregar o arquivo.
    tempo_extracao = float(custo_extrator['tempo_extracao_s'])
    dispositivo = custo_extrator['dispositivo']
    print(f'Features reutilizadas de: {PASTA_FEATURES_SALVAS}')
    print('Tempo, vazão e dispositivo referem-se à extração original.')

# Nova pasta para os resultados desta execução, inclusive quando reutilizamos features.
if RETOMAR_EXECUCAO:
    RUN_DIR=Path(RETOMAR_EXECUCAO)
    EXECUCAO_ID=RUN_DIR.name
else:
    EXECUCAO_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    RUN_DIR=OUTPUT_ROOT/'execucoes'/f'{ARQUITETURA}_{EXECUCAO_ID}'
    RUN_DIR.mkdir(parents=True,exist_ok=False)
custo_extrator['features_reutilizadas'] = not REGERAR_FEATURES
custo_extrator['origem_features'] = str(RUN_DIR) if REGERAR_FEATURES else custo_extrator.get('origem_features', str(PASTA_FEATURES_SALVAS))
print('Features disponíveis; execute a próxima validação antes de continuar.')


In [ ]:
# VALIDAÇÃO 4 — matriz completa antes de salvar os artefatos
assert features.shape == (len(dataset), NUMERO_FEATURES_CNN)
assert features.dtype == np.float32 and np.isfinite(features).all()
np.testing.assert_allclose(features[:len(piloto)], features_piloto, rtol=2e-3, atol=2e-4)
if not RETOMAR_EXECUCAO:
    np.save(RUN_DIR / 'features.npy', features)
    dataset.to_csv(RUN_DIR / 'indice_features.csv', index=False)
    pd.DataFrame([custo_extrator]).to_csv(RUN_DIR / 'custo_extrator.csv', index=False)
display(pd.DataFrame([custo_extrator]).T.rename(columns={0:'valor'}))
print('Artefatos:', RUN_DIR)


## 6. Preparação compartilhada e validação
A primeira célula define funções; não treina. A segunda herda features/base e prepara as divisões. A terceira valida isolamento e salva o plano. Nenhum parâmetro precisa mudar aqui.

**Estático:** aleatório (56% treino / 14% validação / 30% teste), central (teste 35–65%, validação 10% em cada ponta, gaps retirados do treino), extremidade (70/15/15%, retirando gap do início da validação e do teste, como no template anterior), LOEO (7 experimentos treino, 1 validação, 1 teste).

**Temporal:** somente central e comparação aleatória. Central mantém o approach consolidado: quatro cadeias internas, duas subdivisões de cada trecho de treino, com purga do bloco, seus 100 predecessores e gap; uma cadeia externa aprende exclusivamente no treino. Aleatório usa quatro blocos internos por posição e permite contexto visual passado atravessando partições, **nunca rótulos externos**. Continua sujeito à correlação entre frames.

As cadeias mantêm pools fixos; a recursão dentro do pool é in-sample. Isso não é cross-fitting aninhado integral. A divisão e a geração não são reprodução exata do Leonel.

In [ ]:
# Motor comum — divisões, persistência e treinamento estático/temporal.
from sklearn.model_selection import train_test_split
from IPython.display import HTML, clear_output
import ipywidgets as widgets
from html import escape

NOMES_ANALISES = {'aleatorio':'Aleatório (teste 30%)','blocos_central':'Blocos — teste central',
                  'blocos':'Blocos — teste final','loeo':'LOEO'}
ORDEM_ANALISES = list(NOMES_ANALISES)
PROTOCOLOS_TEMPORAIS = ['blocos_central','aleatorio']
PARTICOES = ('treino','validacao','teste')

def digest(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,default=str).encode()).hexdigest()

def gravar_json(path,obj):
    tmp=path.with_suffix('.tmp.json'); tmp.write_text(json.dumps(obj,indent=2,default=str),encoding='utf-8'); tmp.replace(path)

def gravar_npz(path,**dados):
    tmp=path.with_suffix('.tmp.npz'); np.savez(tmp,**dados); tmp.replace(path)

def medir(real,pred):
    real,pred=np.asarray(real).ravel(),np.asarray(pred).ravel()
    rmse=float(np.sqrt(np.mean((real-pred)**2))); desvio=float(np.std(real,ddof=0))
    return {'MAE_mm':float(np.mean(np.abs(real-pred))), 'MAPE_pct':float(100*np.mean(np.abs((real-pred)/real))),
            'RMSE_mm':rmse,'R2':float(1-np.mean((real-pred)**2)/desvio**2) if desvio>0 else np.nan,
            'desvio_y_mm':desvio,'RMSE_desvio':rmse/desvio if desvio>0 else np.nan,
            'vies_mm':float(np.mean(pred-real)),'n':len(real)}

def historicos(base,profundidade,dono=None):
    lookup={(str(e),int(f)):i for i,(e,f) in enumerate(zip(base.experimento,base.frame))}
    h=np.full((len(base),profundidade),-1,dtype=np.int32)
    for i,(e,f) in enumerate(zip(base.experimento,base.frame)):
        for lag in range(1,profundidade+1):
            j=lookup.get((str(e),int(f)-lag),-1)
            if j<0 or (dono is not None and (dono[i]<0 or dono[j]!=dono[i])): break
            h[i,lag-1]=j
    return h

def criar_divisoes(base,protocolo):
    grupos=base.experimento.to_numpy(str); frames=base.frame.to_numpy(int)
    exps=sorted(set(grupos),key=lambda x:int(x[1:]))
    if protocolo=='loeo':
        return [{'nome':exp,'treino':np.flatnonzero((grupos!=exp)&(grupos!=exps[(i-1)%len(exps)])),
                 'validacao':np.flatnonzero(grupos==exps[(i-1)%len(exps)]),'teste':np.flatnonzero(grupos==exp)} for i,exp in enumerate(exps)]
    partes={s:[] for s in PARTICOES}
    for exp in exps:
        ids=np.flatnonzero(grupos==exp); ids=ids[np.argsort(frames[ids],kind='stable')]; n=len(ids)
        if protocolo=='aleatorio':
            restantes,te=train_test_split(ids,test_size=.30,random_state=SEMENTE_DIVISAO)
            tr,va=train_test_split(restantes,test_size=.20,random_state=SEMENTE_DIVISAO+1)
        elif protocolo=='blocos':
            a,b=int(.70*n),int(.85*n)
            tr,va,te=ids[:a],ids[a:b],ids[b:]
            va=va[frames[va]>frames[tr[-1]]+GAP_FRAMES]
            te=te[frames[te]>frames[ids[b-1]]+GAP_FRAMES]
        elif protocolo=='blocos_central':
            a,b,c,d=[int(p*n) for p in (.10,.35,.65,.90)]
            va=np.r_[ids[:a],ids[d:]]; te=ids[b:c]
            esquerda,direita=ids[a:b],ids[c:d]
            esquerda=esquerda[(frames[esquerda]>frames[ids[a-1]]+GAP_FRAMES)&(frames[esquerda]<frames[te[0]]-GAP_FRAMES)]
            direita=direita[(frames[direita]>frames[te[-1]]+GAP_FRAMES)&(frames[direita]<frames[ids[d]]-GAP_FRAMES)]
            tr=np.r_[esquerda,direita]
        else: raise ValueError(protocolo)
        for s,v in zip(PARTICOES,(tr,va,te)): partes[s].append(np.sort(v))
    return [{'nome':'unico',**{s:np.concatenate(v) for s,v in partes.items()}}]

def conferir_divisao(div):
    for s in PARTICOES:
        ids=div[s]
        assert len(ids)>1 and len(np.unique(ids))==len(ids), f'{s}: vazio/duplicado'
        assert np.min(ids)>=0 and np.max(ids)<len(dataset)
    for a,b in [('treino','validacao'),('treino','teste'),('validacao','teste')]:
        assert not np.intersect1d(div[a],div[b]).size, f'Sobreposição: {a}/{b}'

def planejar_temporal(base,div,protocolo):
    dono=np.full(len(base),-1,dtype=int)
    for i,s in enumerate(PARTICOES): dono[div[s]]=i
    # Aleatório: contexto apenas visual pode atravessar partições. Nunca usa seus y.
    hist=historicos(base,max(COLUNAS_TEMPORAIS),None if protocolo=='aleatorio' else dono)
    elegivel=(hist>=0).all(axis=1)
    pareado={s:div[s][elegivel[div[s]]] for s in PARTICOES}
    conferir_divisao(pareado)
    grupos=base.experimento.to_numpy(str); frames=base.frame.to_numpy(int)
    tr=pareado['treino']; planos=[]
    if protocolo=='loeo':
        for exp in sorted(set(grupos[tr])):
            alvo=tr[grupos[tr]==exp]; pool=tr[grupos[tr]!=exp]
            planos.append({'nome':'oof_'+exp,'pool':pool,'alvos':alvo})
    else:
        # Blocos internos por posição; pools contêm os outros experimentos E trechos permitidos do mesmo.
        alvos=[[] for _ in range(4)]
        for exp in sorted(set(grupos[tr])):
            if protocolo=='blocos_central':
                ids=div['treino'][grupos[div['treino']]==exp]; ids=ids[np.argsort(frames[ids])]
                trechos=np.split(ids,np.flatnonzero(np.diff(frames[ids])!=1)+1)
                assert len(trechos)==2,'Teste central exige dois trechos de treino contíguos.'
                for r,trecho in enumerate(trechos):
                    for j,pedaco in enumerate(np.array_split(trecho,2)):
                        alvos[2*r+j].extend(np.intersect1d(pedaco,tr).tolist())
            else:
                ids=tr[grupos[tr]==exp]; ids=ids[np.argsort(frames[ids])]
                for j,pedaco in enumerate(np.array_split(ids,4)): alvos[j].extend(pedaco.tolist())
        for j,lista in enumerate(alvos):
            alvo=np.asarray(lista,dtype=int); permitido=np.ones(len(base),dtype=bool)
            if protocolo=='blocos_central':
                permitido[:]=False; permitido[div['treino']]=True
            for exp in sorted(set(grupos[alvo])):
                a=alvo[grupos[alvo]==exp]
                # Exclui bloco escondido, predecessores utilizados e gap interno.
                lo=frames[a].min()-max(COLUNAS_TEMPORAIS)-GAP_FRAMES
                hi=frames[a].max()+GAP_FRAMES
                permitido &= ~((grupos==exp)&(frames>=lo)&(frames<=hi))
            pool=tr[permitido[tr]&permitido[hist[tr]].all(axis=1)]
            assert len(pool)>=100 and set(grupos[pool])==set(grupos[tr]),'Pool interno insuficiente após purgar histórico.'
            planos.append({'nome':f'oof_{j+1}','pool':pool,'alvos':alvo})
    for p in planos:
        assert len(p['pool'])>1 and len(p['alvos'])>0,'Pool auxiliar insuficiente; rever divisão, não reduzir gap silenciosamente.'
        assert not np.intersect1d(p['pool'],p['alvos']).size
        assert np.isin(p['pool'],div['treino']).all()
        assert not np.intersect1d(p['pool'],hist[p['alvos']].ravel()).size
        assert not np.intersect1d(hist[p['pool']].ravel(),p['alvos']).size
    assert np.array_equal(np.sort(np.concatenate([p['alvos'] for p in planos])),np.sort(tr))
    planos.append({'nome':'externa','pool':tr,'alvos':np.r_[pareado['validacao'],pareado['teste']]})
    return {'div':pareado,'hist':hist,'planos':planos}

def construir_modelo(n,seed):
    keras.backend.clear_session(); keras.utils.set_random_seed(int(seed))
    m=keras.Sequential([keras.layers.Input(shape=(n,)),keras.layers.Dense(32,activation='relu'),keras.layers.Dense(1)])
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=TAXA_APRENDIZADO),loss='mae'); return m

def preparar_diretorio(path,identidade):
    path.mkdir(parents=True,exist_ok=True)
    arq=path/'manifesto.json'
    if arq.exists():
        if json.loads(arq.read_text())!=identidade: raise ValueError(f'Checkpoint incompatível: {path}')
    else: gravar_json(arq,identidade)

def ajustar_final(path,div,seed,k=0,temporal=None,custo_aux=0.):
    if (path/'concluido.json').exists(): return
    path.mkdir(parents=True,exist_ok=True)
    def entrada(ids): return X[ids] if k==0 else np.column_stack([X[ids],temporal[ids,:k]])
    tr,va=div['treino'],div['validacao']; scaler=StandardScaler().fit(entrada(tr))
    m=construir_modelo(entrada(tr).shape[1],seed)
    inicio=perf_counter()
    h=m.fit(scaler.transform(entrada(tr)).astype('float32'),y[tr],validation_data=(scaler.transform(entrada(va)).astype('float32'),y[va]),
            epochs=EPOCAS,batch_size=BATCH_REGRESSAO,verbose=0,shuffle=True,
            callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss',patience=PACIENCIA,restore_best_weights=True)])
    fit=perf_counter()-inicio; linhas=[]; partes=[]
    for s in PARTICOES:
        ids=div[s]; pred=m.predict(scaler.transform(entrada(ids)).astype('float32'),batch_size=BATCH_REGRESSAO,verbose=0).ravel()
        assert np.isfinite(pred).all()
        linhas.append({'conjunto':s,'colunas':k,'semente':seed,**medir(y[ids],pred),
                       'tempo_fit_final_s':fit,'tempo_auxiliar_acumulado_s':custo_aux,'epocas':len(h.history['loss'])})
        p=dataset.iloc[ids][['indice_dataset','experimento','frame','y']].copy()
        p['y_predito']=pred; p['conjunto']=s; partes.append(p)
    m.save(path/'modelo.keras'); gravar_npz(path/'scaler.npz',media=scaler.mean_,escala=scaler.scale_)
    pd.DataFrame(h.history).to_csv(path/'historico.csv',index=False)
    pd.DataFrame(linhas).to_csv(path/'metricas.csv',index=False); pd.concat(partes).to_csv(path/'predicoes.csv',index=False)
    gravar_json(path/'concluido.json',{'K':k,'semente':seed})

def gerar_temporal(path,plano,seed):
    prof=max(COLUNAS_TEMPORAIS); hist=plano['hist']; temporal=np.full((len(dataset),prof),np.nan,dtype='float32'); custos=np.zeros(prof)
    for numero,p in enumerate(plano['planos']):
        local=path/p['nome']; local.mkdir(parents=True,exist_ok=True)
        pool,alvos=p['pool'],p['alvos']; gravar_npz(local/'indices.npz',pool=pool,alvos=alvos)
        cols=np.full_like(temporal,np.nan); tempos=np.zeros(prof); etapa=0
        if (local/'checkpoint.npz').exists():
            with np.load(local/'checkpoint.npz',allow_pickle=False) as z: cols=z['colunas'].copy(); tempos=z['tempos'].copy(); etapa=int(z['etapa'])
            assert cols.shape==temporal.shape and tempos.shape==(prof,) and 0<=etapa<=prof
            assert np.isfinite(cols[np.r_[pool,alvos],:etapa]).all()
        print(f'{path.name} | {p["nome"]}: retomada {etapa}/{prof}',flush=True)
        def entrada(ids,k): return X[ids] if k==0 else np.column_stack([X[ids],cols[ids,:k]])
        for k in range(etapa,prof):
            inicio=perf_counter(); xt=entrada(pool,k); assert np.isfinite(xt).all()
            scaler=StandardScaler().fit(xt); m=construir_modelo(xt.shape[1],seed+1000*(numero+1)+k)
            m.fit(scaler.transform(xt).astype('float32'),y[pool],epochs=EPOCAS_AUX,batch_size=BATCH_REGRESSAO,verbose=0,shuffle=True)
            destinos=np.flatnonzero((hist[:,:k+1]>=0).all(axis=1)); fontes=hist[destinos,0]
            xp=entrada(fontes,k); assert np.isfinite(xp).all()
            pred=m.predict(scaler.transform(xp).astype('float32'),batch_size=BATCH_REGRESSAO,verbose=0).ravel(); assert np.isfinite(pred).all()
            cols[destinos,k]=pred; tempos[k]=perf_counter()-inicio
            m.save(local/'ultimo_auxiliar.keras')
            gravar_npz(local/'checkpoint.npz',colunas=cols,tempos=tempos,etapa=np.array(k+1))
            if k==0 or (k+1)%10==0: print(f'  {p["nome"]}: coluna {k+1}/{prof}',flush=True)
        temporal[alvos]=cols[alvos]; custos+=tempos
    assert np.isfinite(temporal[np.concatenate(list(plano['div'].values()))]).all()
    np.save(path/'features_temporais.npy',temporal)
    pd.DataFrame({'coluna':np.arange(1,prof+1),'tempo_auxiliar_s':custos}).to_csv(path/'custos_auxiliares.csv',index=False)
    return temporal,np.cumsum(custos)

def executar_etapa(etapa):
    assert CONSOLIDADO_VALIDADO,'Execute a validação primeiro.'
    atuais={'arquitetura':ARQUITETURA,'parametros_processo':USAR_PARAMETROS,'sementes':list(SEMENTES),
        'colunas':list(COLUNAS_TEMPORAIS),'gap_frames':GAP_FRAMES,'epocas_aux':EPOCAS_AUX,
        'epocas':EPOCAS,'paciencia':PACIENCIA,'batch_regressao':BATCH_REGRESSAO,'taxa_aprendizado':TAXA_APRENDIZADO}
    if any(config_analise[k]!=v for k,v in atuais.items()):
        raise ValueError('Configuração alterada após validar. Prepare e valide novamente antes de treinar.')
    for protocolo in (ORDEM_ANALISES if etapa=='estatico' else PROTOCOLOS_TEMPORAIS):
        for seed in SEMENTES:
            for numero,div in enumerate(protocolos[protocolo]):
                plano=planos_temporais.get((protocolo,div['nome']))
                pasta=RUN_DIR/etapa/protocolo/f'seed_{seed}'/div['nome']
                identidade={'assinatura':ASSINATURA_EXECUCAO,'etapa':etapa,'protocolo':protocolo,'seed':seed,'fold':div['nome'],'tensorflow':tf.__version__}
                preparar_diretorio(pasta,identidade)
                if (pasta/'concluido.json').exists(): print('Reutilizado:',pasta); continue
                print(f'{etapa} | {protocolo} | seed {seed} | {div["nome"]}',flush=True)
                if etapa=='estatico': ajustar_final(pasta/'K_000',div,seed+numero)
                else:
                    # K=0 é referência pareada, não substitui o estático com todos os exemplos.
                    ajustar_final(pasta/'K_000',plano['div'],seed+numero)
                    temporal,custos=gerar_temporal(pasta,plano,seed+numero)
                    for k in COLUNAS_TEMPORAIS[1:]:
                        ajustar_final(pasta/f'K_{k:03d}',plano['div'],seed+numero,k,temporal,float(custos[k-1]))
                    vals=[pd.read_csv(pasta/f'K_{k:03d}'/'metricas.csv').query("conjunto == 'validacao'").iloc[0] for k in COLUNAS_TEMPORAIS]
                    escolhido=min(vals,key=lambda r:(r.MAE_mm,r.colunas))
                    gravar_json(pasta/'escolha_validacao.json',{'K':int(escolhido.colunas),'criterio':'menor MAE de validação; empate favorece menor K'})
                gravar_json(pasta/'concluido.json',identidade)
    print('Etapa concluída:',etapa)

# Painéis compartilhados — leem somente a nova pasta de execuções, sem treinar.
def carregar_resultados(raiz):
    linhas=[]; avisos=[]
    for arquivo in sorted(Path(raiz).glob('*/manifesto.json')):
        try:
            m=json.loads(arquivo.read_text()); cfg=m['config']; run=arquivo.parent
            for etapa in ['estatico','temporal']:
                protos=cfg['protocolos'] if etapa=='estatico' else cfg['protocolos_temporais']
                ks=[0] if etapa=='estatico' else cfg['colunas']
                for proto in protos:
                    for seed in cfg['sementes']:
                        pasta=run/etapa/proto/f'seed_{seed}'
                        esperados=sorted(pd.read_csv(run/'indice_features.csv').experimento.unique()) if proto=='loeo' else ['unico']
                        folds=[pasta/f for f in esperados]
                        if not all((f/'concluido.json').exists() for f in folds):
                            avisos.append(f'{run.name} | {etapa}/{proto}/seed {seed}: pendente'); continue
                        for f in folds:
                            mf=json.loads((f/'manifesto.json').read_text())
                            if mf['assinatura']!=m['assinatura'] or json.loads((f/'concluido.json').read_text())!=mf:
                                raise ValueError('Assinatura divergente: '+str(f))
                        for k in ks:
                            pred=[]; metric=[]
                            for f in folds:
                                fp=f/f'K_{k:03d}'
                                if not (fp/'concluido.json').exists(): raise ValueError('K incompleto: '+str(fp))
                                d=pd.read_csv(fp/'predicoes.csv'); d['fold']=f.name; pred.append(d)
                                metric.append(pd.read_csv(fp/'metricas.csv'))
                            d=pd.concat(pred,ignore_index=True); metr=pd.concat(metric,ignore_index=True)
                            for conjunto,g in d.groupby('conjunto'):
                                if conjunto=='teste' and g.duplicated('indice_dataset').any(): raise ValueError('Teste duplicado.')
                                if not np.isfinite(g[['y','y_predito']]).all().all(): raise ValueError('Previsões inválidas.')
                                custos=metr.loc[metr.conjunto==conjunto]
                                linhas.append({'execucao':run.name,'pasta':str(run),'CNN':cfg['arquitetura'],
                                    'base':cfg['dataset_id'],'grupo':m['grupo'],'parametros':cfg['parametros_processo'],
                                    'etapa':etapa,'protocolo':proto,'semente':seed,'colunas':k,'conjunto':conjunto,
                                    **medir(g.y,g.y_predito),'custo_s':float((custos.tempo_fit_final_s+custos.tempo_auxiliar_acumulado_s).sum())})
        except (KeyError,ValueError,OSError) as exc: avisos.append(f'{arquivo}: {exc}')
    return pd.DataFrame(linhas),avisos

def painel_comparativo(etapa):
    historico,avisos=carregar_resultados(OUTPUT_ROOT/'execucoes')
    if historico.empty or not (historico.etapa==etapa).any():
        print('Ainda não há resultados completos desta etapa na nova pasta. Execute novamente esta célula após os treinos.'); return
    h=historico.loc[historico.etapa==etapa].copy()
    grupos=[(f'Base {g.base.iloc[0]} | parâmetros={g.parametros.iloc[0]} | {key[:10]}',key) for key,g in h.groupby('grupo',sort=False)]
    base_w=widgets.Dropdown(options=grupos,description='Base/grupo:',layout=widgets.Layout(width='98%'))
    metodo_w=widgets.Dropdown(options=[(NOMES_ANALISES[p],p) for p in h.protocolo.unique()],description='Método:',layout=widgets.Layout(width='340px'))
    exec_w=widgets.SelectMultiple(description='Execuções:',rows=5,layout=widgets.Layout(width='98%'))
    seed_w=widgets.Dropdown(options=['Média 42/43','42','43'],description='Sementes:')
    split_w=widgets.Dropdown(options=['teste','validacao','treino'],description='Conjunto:')
    k_w=widgets.Dropdown(options=['Todos']+list(COLUNAS_TEMPORAIS) if etapa=='temporal' else [0],value=0,description='K (barras):')
    met_w=widgets.Dropdown(options=[('MAE (mm) ↓','MAE_mm'),('MAPE (%) ↓','MAPE_pct'),('RMSE (mm) ↓','RMSE_mm'),('R² ↑','R2'),('RMSE/desvio ↓','RMSE_desvio')],description='Métrica:')
    tipo_w=widgets.Dropdown(options=['Evolução do MAE','Variação percentual','Barras','Custo × MAE','RMSE/desvio'],value='Evolução do MAE' if etapa=='temporal' else 'Barras',description='Gráfico:',layout=widgets.Layout(width='330px'))
    out=widgets.Output(); atualizar=widgets.Button(description='Atualizar gráfico',icon='refresh')
    def preencher(_=None):
        d=h.loc[h.grupo==base_w.value]; runs=d[['execucao','CNN','parametros']].drop_duplicates()
        exec_w.options=[(f'{r.CNN} | parâmetros={r.parametros} | {r.execucao}',r.execucao) for r in runs.itertuples()]
        exec_w.value=tuple(runs.execucao)
    def desenhar(_=None):
        with out:
            clear_output(wait=True)
            d=h.loc[(h.grupo==base_w.value)&(h.protocolo==metodo_w.value)&(h.conjunto==split_w.value)&h.execucao.isin(exec_w.value)].copy()
            if seed_w.value=='Média 42/43':
                d=d.groupby(['execucao','colunas'],group_keys=False).filter(lambda g:set(g.semente)=={42,43})
            else: d=d.loc[d.semente==int(seed_w.value)]
            if d.empty: print('Sem resultados completos para estes filtros.'); return
            partes=[]
            for _,g in d.groupby(['execucao','semente']):
                g=g.sort_values('colunas').copy(); ref=g.loc[g.colunas==0,'MAE_mm']
                if len(ref)!=1: raise ValueError('Referência K=0 ausente/duplicada.')
                g['vs_estatico_pct']=100*(g.MAE_mm/float(ref.iloc[0])-1)
                g['vs_anterior_pct']=100*(g.MAE_mm/g.MAE_mm.shift(1)-1); partes.append(g)
            d=pd.concat(partes)
            metricas=['MAE_mm','MAPE_pct','RMSE_mm','R2','RMSE_desvio','custo_s','vs_estatico_pct','vs_anterior_pct']
            r=d.groupby(['execucao','CNN','parametros','colunas'],as_index=False)[metricas].mean()
            print(f'{etapa.upper()} | {NOMES_ANALISES[metodo_w.value]} | {split_w.value} | {seed_w.value}')
            print('Média 42/43 = média das métricas por semente (não previsões agrupadas). Variações percentuais calculadas por semente antes da média.')
            if tipo_w.value in ['Evolução do MAE','Variação percentual']:
                cols=['MAE_mm'] if tipo_w.value=='Evolução do MAE' else ['vs_estatico_pct','vs_anterior_pct']
                fig,axes=plt.subplots(1,len(cols),figsize=(7*len(cols),4.5),squeeze=False)
                for ax,col in zip(axes.flat,cols):
                    for (run,cnn,param),g in r.groupby(['execucao','CNN','parametros']):
                        g=g.sort_values('colunas'); ax.plot(range(len(g)),g[col],'o-',label=f'{cnn} | parâmetros={param} | {run}')
                        ax.set_xticks(range(len(g)),g.colunas)
                    ax.set_xlabel('Colunas temporais K'); ax.set_ylabel(col); ax.grid(alpha=.2)
                    if col!='MAE_mm': ax.axhline(0,color='gray',lw=.8)
                axes.flat[0].legend(fontsize=7,bbox_to_anchor=(0,-.25),loc='upper left')
            else:
                if k_w.value!='Todos': r=r.loc[r.colunas==int(k_w.value)]
                r=r.sort_values(met_w.value,ascending=met_w.value!='R2').reset_index(drop=True)
                if r.empty: print('Sem K selecionado.'); return
                r['ID']=[f'E{i+1:02d}' for i in range(len(r))]
                labels=[f'{a.ID} | {a.CNN} | parâmetros={a.parametros} | K={a.colunas} | {a.execucao}' for a in r.itertuples()]
                fig,ax=plt.subplots(figsize=(12,max(4,.45*len(r)+1.5)))
                if tipo_w.value=='Custo × MAE':
                    from matplotlib.colors import Normalize
                    valid=r.loc[(r.custo_s>0)&np.isfinite(r.R2)]
                    if valid.empty: plt.close(fig); print('Custo/R² indisponíveis.'); return
                    norm=Normalize(vmin=min(0,valid.R2.min()),vmax=max(1,valid.R2.max()))
                    for param,g in valid.groupby('parametros'):
                        sc=ax.scatter(g.custo_s,g.MAE_mm,c=g.R2,cmap='viridis',norm=norm,marker='D' if param else 'o',s=90,edgecolors='gray',label=f'parâmetros={param}')
                        for a in g.itertuples(): ax.annotate(f'{a.ID} · K={a.colunas}',(a.custo_s,a.MAE_mm),xytext=(5,6),textcoords='offset points',fontsize=8)
                    ax.set_xscale('log'); ax.set_xlabel('Auxiliares até K + treino final (s; sem extração CNN)'); ax.set_ylabel('MAE (mm)')
                    fig.colorbar(sc,ax=ax,label='R² médio' if seed_w.value=='Média 42/43' else 'R²'); ax.legend()
                else:
                    col='RMSE_desvio' if tipo_w.value=='RMSE/desvio' else met_w.value
                    bars=ax.barh(range(len(r)),r[col],color=['#d17b21' if p else '#386cb0' for p in r.parametros])
                    ax.set_yticks(range(len(r)),labels,fontsize=8); ax.invert_yaxis(); ax.bar_label(bars,fmt='%.4f',padding=4)
                    ax.axvline(1 if col=='RMSE_desvio' else 0,color='gray',lw=1); ax.margins(x=.2); ax.set_title(col)
                ax.grid(alpha=.2); ax.set_axisbelow(True)
            fig.tight_layout(); plt.show(); plt.close(fig)
            display(HTML('<details><summary>Valores e identificação das execuções</summary>'+r.to_html(index=False,float_format=lambda v:f'{v:.4f}',escape=True)+'</details>'))
            print('Custos são tempos registrados, não benchmark de hardware controlado; excluem extração CNN e busca de outros K.')
            print('RMSE/desvio > 1: erro maior que a dispersão dos alvos; não prova isoladamente pouca variabilidade.')
            if metodo_w.value=='aleatorio': print('Aleatório: frames correlacionados e histórico visual compartilhado podem tornar o resultado otimista.')
            print('Escolha de K usa validação. O teste não deve orientar ajustes; conjuntos diferem entre protocolos.')
    preencher(); base_w.observe(preencher,names='value')
    for w in [base_w,metodo_w,exec_w,seed_w,split_w,k_w,met_w,tipo_w]: w.observe(desenhar,names='value')
    atualizar.on_click(desenhar)
    display(widgets.VBox([base_w,exec_w,widgets.HBox([metodo_w,seed_w,split_w]),widgets.HBox([tipo_w,k_w,met_w]),atualizar,out])); desenhar()

def resumo_atual(etapa):
    d,avisos=carregar_resultados(OUTPUT_ROOT/'execucoes')
    if d.empty: print('Nenhuma etapa completa salva.'); return
    d=d.loc[(d.pasta==str(RUN_DIR))&(d.etapa==etapa)&(d.conjunto=='teste')]
    if d.empty: print('Resultados ainda não concluídos nesta execução.'); return
    r=d.groupby(['protocolo','colunas'])[['MAE_mm','MAPE_pct','RMSE_mm','R2']].mean()
    display(r.style.format('{:.4f}').set_properties(subset=['MAE_mm'],**{'font-weight':'bold','background-color':'#eaf2ff'}))
    print('Médias entre sementes disponíveis; para comparação final use somente pares 42/43 completos no painel.')

def painel_curvas():
    fontes=[]
    for arq in sorted((OUTPUT_ROOT/'execucoes').glob('*/manifesto.json')):
        m=json.loads(arq.read_text()); cfg=m['config']
        for p in cfg['protocolos_temporais']:
            for seed in cfg['sementes']:
                local=arq.parent/'temporal'/p/f'seed_{seed}'/'unico'
                if (local/'features_temporais.npy').exists(): fontes.append((f'{cfg["arquitetura"]} | {p} | seed={seed} | {arq.parent.name}',str(local)))
    if not fontes: print('As curvas estarão disponíveis após a geração temporal.'); return
    fonte=widgets.Dropdown(options=fontes,description='Execução:',layout=widgets.Layout(width='98%'))
    exp=widgets.Dropdown(options=EXPERIMENTOS,description='Experimento:')
    coluna=widgets.IntSlider(value=1,min=1,max=max(COLUNAS_TEMPORAIS),description='Coluna:',continuous_update=False)
    out=widgets.Output()
    def desenhar(_=None):
        with out:
            clear_output(wait=True); local=Path(fonte.value); run=local.parents[3]; proto=local.parents[1].name
            base=pd.read_csv(run/'indice_features.csv'); t=np.load(local/'features_temporais.npy',mmap_mode='r',allow_pickle=False)
            with np.load(run/'divisoes'/proto/'unico'/'pareado.npz',allow_pickle=False) as z: div={s:z[s] for s in PARTICOES}
            fig,axes=plt.subplots(3,1,figsize=(13,8),sharex=True)
            for ax,s in zip(axes,PARTICOES):
                ids=div[s]; ids=ids[base.iloc[ids].experimento.to_numpy()==exp.value]; ids=ids[np.argsort(base.iloc[ids].frame)]
                ax.scatter(base.iloc[ids].frame,base.iloc[ids].y,s=5,label='y do frame atual',color='#17324d')
                ax.scatter(base.iloc[ids].frame,t[ids,coluna.value-1],s=4,alpha=.6,label=f'Feature temporal {coluna.value}',color='#cc8b38')
                ax.set_title(s,loc='left'); ax.set_ylabel('Largura (mm)'); ax.grid(alpha=.2); ax.legend(fontsize=8)
            axes[-1].set_xlabel('Número do frame'); fig.suptitle(f'{NOMES_ANALISES[proto]} | {exp.value} | geração iterativa')
            fig.tight_layout(); plt.show(); plt.close(fig)
            print('A coluna j é previsão de t−1 de um auxiliar com j−1 colunas; não é simplesmente a medição de t−j.')
    for w in [fonte,exp,coluna]: w.observe(desenhar,names='value')
    display(widgets.VBox([fonte,widgets.HBox([exp,coluna]),out])); desenhar()


In [ ]:
# Preparação — herda a extração e a base desta execução, sem selecionar outro arquivo.
CONSOLIDADO_VALIDADO=False
if RETOMAR_EXECUCAO:
    RUN_DIR=Path(RETOMAR_EXECUCAO)
    dataset=pd.read_csv(RUN_DIR/'indice_features.csv')
    features=np.load(RUN_DIR/'features.npy',mmap_mode='r',allow_pickle=False)
    custo_extrator=pd.read_csv(RUN_DIR/'custo_extrator.csv').iloc[0].to_dict()
    origem=json.loads((RUN_DIR/'configuracao.json').read_text())
    DATASET_ID=origem['dataset_id']; TAMANHO_ENTRADA=tuple(origem['entrada'])
    # A base Sofia foi criada com y float32; o CSV não preserva esse tipo.
    # Restaura o tipo original antes da assinatura, sem relaxar sua verificação.
    if not origem['dataset_antigo']:
        dataset['y']=dataset['y'].astype(np.float32)
    if origem['arquitetura']!=ARQUITETURA or origem['parametros_processo']!=USAR_PARAMETROS or origem['pooling']!=POOLING:
        raise ValueError('Configuração atual diferente da execução a retomar.')
X=np.asarray(features,dtype=np.float32)
if USAR_PARAMETROS: X=np.column_stack([X,dataset[COLUNAS_PARAMETROS].to_numpy(dtype=np.float32)])
y=dataset.y.to_numpy(dtype=np.float32)
protocolos={p:criar_divisoes(dataset,p) for p in ORDEM_ANALISES}
planos_temporais={}
for p,folds in protocolos.items():
    if p in PROTOCOLOS_TEMPORAIS:
        for f in folds: planos_temporais[(p,f['nome'])]=planejar_temporal(dataset,f,p)
config_analise={'versao':'consolidado_v1','dataset_id':DATASET_ID,'arquitetura':ARQUITETURA,
    'parametros_processo':USAR_PARAMETROS,'sementes':list(SEMENTES),'semente_divisao':SEMENTE_DIVISAO,
    'colunas':list(COLUNAS_TEMPORAIS),'gap_frames':GAP_FRAMES,'epocas_aux':EPOCAS_AUX,'epocas':EPOCAS,
    'paciencia':PACIENCIA,'batch_regressao':BATCH_REGRESSAO,'taxa_aprendizado':TAXA_APRENDIZADO,
    'regressor':'Dense(32,relu)-Dense(1)','tensorflow':tf.__version__,'origem_features':str(RUN_DIR),
    'pooling':POOLING,'entrada':list(TAMANHO_ENTRADA),'protocolos':ORDEM_ANALISES,'protocolos_temporais':PROTOCOLOS_TEMPORAIS,
    'dataset_antigo':USAR_DATASET_ANTIGO,
    'historico_aleatorio':'imagens passadas podem atravessar partições; y auxiliar apenas de treino',
    'auxiliares':'pool fixo por cadeia; exclusão de bloco+histórico+gap; recursão interna in-sample, não cross-fitting aninhado completo'}
hash_base=hashlib.sha256(dataset[['experimento','frame','caminho_relativo','y']+COLUNAS_PARAMETROS].to_csv(index=False,float_format='%.9g').encode()).hexdigest()
hash_x=hashlib.sha256(np.ascontiguousarray(X).tobytes()).hexdigest()
div_ids={p:[{s:f[s].tolist() for s in PARTICOES} for f in folds] for p,folds in protocolos.items()}
comum={k:v for k,v in config_analise.items() if k not in ['arquitetura','origem_features','pooling','entrada']}
GRUPO_COMPARACAO=digest({'base':hash_base,'divisoes':div_ids,'config':comum})[:16]
ASSINATURA_EXECUCAO=digest({'config':config_analise,'base':hash_base,'X':hash_x,'divisoes':div_ids})
manifesto_execucao={'config':config_analise,'assinatura':ASSINATURA_EXECUCAO,'grupo':GRUPO_COMPARACAO,
    'base_hash':hash_base,'features_hash':hash_x}
preparar_diretorio(RUN_DIR,manifesto_execucao)
gravar_json(RUN_DIR/'configuracao.json',config_analise)
print('Execução:',RUN_DIR,'| grupo comparável:',GRUPO_COMPARACAO)



In [ ]:
# VALIDAÇÃO separada — sem treinamento.

assert np.array_equal(dataset.indice_dataset,np.arange(len(dataset)))
assert not dataset.duplicated(['experimento','frame']).any()
assert np.array_equal(dataset.frame.to_numpy(),dataset.frame.to_numpy(dtype=int))
assert np.isfinite(X).all() and np.isfinite(y).all() and (y>0).all()
pd.testing.assert_frame_equal(dataset.reset_index(drop=True),pd.read_csv(RUN_DIR/'indice_features.csv').reset_index(drop=True),check_dtype=False,rtol=1e-6,atol=1e-6)
linhas=[]; total_fits=0
for p,folds in protocolos.items():
    for f in folds:
        conferir_divisao(f)
        if p=='loeo':
            gs=[set(dataset.iloc[f[s]].experimento) for s in PARTICOES]
            assert not gs[0]&gs[1] and not gs[0]&gs[2] and not gs[1]&gs[2]
        pasta=RUN_DIR/'divisoes'/p/f['nome']; pasta.mkdir(parents=True,exist_ok=True)
        gravar_npz(pasta/'estatico.npz',**{s:f[s] for s in PARTICOES})
        if p not in PROTOCOLOS_TEMPORAIS:
            continue
        plano=planos_temporais[(p,f['nome'])]; conferir_divisao(plano['div'])
        if p!='aleatorio':
            for s in PARTICOES:
                assert np.isin(plano['hist'][plano['div'][s]],f[s]).all()
        pasta=RUN_DIR/'divisoes'/p/f['nome']; pasta.mkdir(parents=True,exist_ok=True)
        gravar_npz(pasta/'estatico.npz',**{s:f[s] for s in PARTICOES})
        gravar_npz(pasta/'pareado.npz',**plano['div'])
        fits=len(SEMENTES)*(len(plano['planos'])*max(COLUNAS_TEMPORAIS)+len(COLUNAS_TEMPORAIS))
        total_fits+=fits
        linhas.append({'protocolo':NOMES_ANALISES[p],'fold':f['nome'],**{s:len(plano['div'][s]) for s in PARTICOES},'fits_temporais':fits})
display(pd.DataFrame(linhas))
CONSOLIDADO_VALIDADO=True
print(f'Validado. Temporal: {total_fits:,} fits planejados; estático: {len(SEMENTES)*sum(map(len,protocolos.values()))}.')
print('Execução longa — checkpoints por coluna e por regressor. Não iniciada por esta célula.')


## 7. Executar o estático — quatro protocolos
Ative `EXECUTAR_ESTATICO=True` na configuração. Usa sementes 42/43 e validação independente em todos os protocolos. Há checkpoints por regressor; uma etapa concluída é reutilizada. Não reextrai as features.

In [ ]:
if EXECUTAR_ESTATICO:
    executar_etapa('estatico')
else:
    print('Estático desativado. Defina EXECUTAR_ESTATICO=True na configuração e execute a preparação/validação antes de treinar.')

## 8. Resultados estáticos
Lê a execução atual. MAE destacado; detalhes e métricas globais/perfis são acessíveis nos painéis. No LOEO o R² de teste é calculado juntando previsões fora do treino, não pela média dos R² por experimento.

In [ ]:
resumo_atual('estatico')

## 9. Comparação estática entre execuções salvas
Selecione grupo compatível, execuções (Ctrl/Cmd para múltiplas), protocolo, semente e gráfico. Preserva barras, custo × MAE e RMSE/desvio. Custo representa treinamento registrado, sem extração CNN; comparar hardware diferente exige cautela. Execute novamente para reler a pasta.

In [ ]:
# Lê execuções salvas; não depende dos modelos em memória.
painel_comparativo('estatico')

## 10. Temporal consolidado — executar



Ative `EXECUTAR_TEMPORAL=True` após validar o plano. **Somente bloco central e comparação com divisão aleatória; sem LOEO temporal nem teste na extremidade.**

Gera 100 colunas iterativamente com retreinamento auxiliar; avalia finais K=0/1/5/10/25/50/100, sementes 42/43. A coluna j contém previsão de t−1 produzida por modelo com j−1 colunas, não simplesmente a imagem t−j. Frames sem histórico ficam NaN e são excluídos da comparação pareada (não preenchidos com y).

Todos os K usam os mesmos exemplos. Auxiliares têm 50 épocas fixas; finais usam EarlyStopping na validação; scaler somente no treino. **2.028 fits temporais por CNN**. Checkpoint a cada coluna; salvamento de métricas, previsões, modelo final e scaler. K escolhido pelo MAE de validação, não pelo teste.

In [ ]:
if EXECUTAR_TEMPORAL:
    executar_etapa('temporal')
else:
    print('Temporal desativado. Defina EXECUTAR_TEMPORAL=True após conferir a validação. Apenas bloco central e aleatório; sem LOEO.')

## 11. Resultados temporais da execução
Quadro de métricas por protocolo e K. Compare com K=0 do mesmo protocolo, não com o estático que usa mais frames. Duas sementes dão uma verificação descritiva, não significância estatística.

In [ ]:
resumo_atual('temporal')

## 12. Curvas e comparação temporal entre execuções
Primeira célula: selecione execução/protocolo/semente, experimento e coluna; visualize y e feature em treino, validação e teste.

Segunda célula: escolha uma ou várias execuções para comparar CNNs na mesma base/configuração. Gráficos: evolução do MAE, variação frente a K=0 e K anterior, barras, custo × MAE/R² e RMSE/desvio. O filtro K atua nas barras/dispersão; a evolução mostra todos os sete K. Valores negativos de variação significam redução do MAE. O título sempre informa os filtros aplicados.

In [ ]:
painel_curvas()

In [ ]:
painel_comparativo('temporal')

## 13. Persistência e retomada
Cada pasta de execução guarda features/índice/custo, manifesto e assinaturas, divisões, subpastas `estatico` e `temporal`, sementes, protocolos e checkpoints. Painéis leem exclusivamente essa nova raiz. **Não limpe a pasta durante execução** e não altere parâmetros na retomada. Os resultados antigos permanecem na raiz anterior.

In [ ]:
print('Resultados:', RUN_DIR)
print('Para retomar: copie esse caminho em RETOMAR_EXECUCAO; rode configuração, funções, preparação e validação. Execute apenas a etapa pendente.')
print('Para atualizar gráficos após novos treinos, reexecute as células dos painéis. Nenhum resultado antigo foi excluído.')

In [ ]:
#@title 14. Painéis adicionais — custos, ganho temporal e sementes (somente leitura)
# Requer a célula de funções compartilhadas. Não treina nem modifica resultados.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from pathlib import Path
from IPython.display import display, clear_output

def painel_adicional_tcc():
    if 'carregar_resultados' not in globals():
        print('Execute primeiro a célula de funções da seção 6, sem iniciar os treinamentos.'); return
    h, avisos = carregar_resultados(OUTPUT_ROOT / 'execucoes')
    if h.empty:
        print('Nenhuma execução completa disponível.'); return
    grupo = widgets.Dropdown(options=[(f'{g.base.iloc[0]} | parâmetros={g.parametros.iloc[0]} | {key}', key) for key,g in h.groupby('grupo')], description='Base/grupo:', layout=widgets.Layout(width='98%'))
    runs = widgets.SelectMultiple(description='Execuções:', rows=5, layout=widgets.Layout(width='98%'))
    proto = widgets.Dropdown(options=sorted(h.protocolo.unique()), description='Protocolo:')
    seed = widgets.Dropdown(options=[42,43], description='Semente:')
    kval = widgets.Dropdown(options=[0,1,5,10,25,50,100], value=10, description='K:')
    modo = widgets.Dropdown(options=['Extração por CNN','Custo temporal por etapa','Custo acumulado por K','Ganho temporal por experimento','Comparação entre sementes'], description='Visual:', layout=widgets.Layout(width='450px'))
    out = widgets.Output()
    def preencher(_=None):
        d=h[h.grupo==grupo.value][['execucao','CNN']].drop_duplicates()
        runs.options=[(f'{r.CNN} | {r.execucao}',r.execucao) for r in d.itertuples()]
        runs.value=tuple(d.execucao)
    def desenhar(_=None):
        with out:
            clear_output(wait=True)
            d=h[(h.grupo==grupo.value)&h.execucao.isin(runs.value)&(h.protocolo==proto.value)]
            if d.empty: print('Sem execuções completas para os filtros.'); return
            print('Grupo compatível; hardware de treinamento pode diferir entre sessões. Tempos registrados não são benchmark controlado.')
            try:
                if modo.value=='Extração por CNN':
                    rows=[]
                    for pasta in d.pasta.unique():
                        a=pd.read_csv(Path(pasta)/'custo_extrator.csv').iloc[0].to_dict()
                        a['origem']=a.get('origem_features',pasta); rows.append(a)
                    r=pd.DataFrame(rows).drop_duplicates('origem')
                    labels=[f"{a.arquitetura} | {a.dispositivo} | lote {a.batch_extracao}" for a in r.itertuples()]
                    fig,axs=plt.subplots(1,2,figsize=(14,max(4,len(r)*.6)))
                    for ax,col,fator,titulo in zip(axs,['tempo_extracao_s','ms_imagem_lote'],[1/60,1],['Extração (min)','Tempo amortizado (ms/imagem)']):
                        ax.barh(labels,r[col]*fator,color='#44779c'); ax.set_xlabel(titulo); ax.grid(axis='x',alpha=.2)
                    fig.tight_layout(); plt.show(); plt.close(fig)
                    display(r[['arquitetura','dispositivo','batch_extracao','entrada','dtype','memoria_features_MiB','origem']])
                    print('Inclui leitura/pré-processamento/CNN. Extrações reutilizadas contadas uma vez por origem. Não mede latência em tempo real.'); return
                t=d[(d.etapa=='temporal')&(d.semente==seed.value)]
                if modo.value in ['Custo temporal por etapa','Custo acumulado por K']:
                    rows=[]
                    for a in t[t.conjunto=='teste'].itertuples():
                        path=Path(a.pasta)/'temporal'/a.protocolo/f'seed_{a.semente}'/'unico'/f'K_{a.colunas:03d}'/'metricas.csv'
                        m=pd.read_csv(path); m=m[m.conjunto=='teste'].iloc[0]
                        rows.append(dict(execucao=a.execucao,CNN=a.CNN,K=a.colunas,aux=m.tempo_auxiliar_acumulado_s/60,final=m.tempo_fit_final_s/60))
                    r=pd.DataFrame(rows)
                    if r.empty: print('Sem custos temporais completos.'); return
                    fig,ax=plt.subplots(figsize=(12,5))
                    if modo.value=='Custo temporal por etapa':
                        r=r[r.K==kval.value]
                        if r.empty: plt.close(fig); print('K indisponível.'); return
                        labels=[f'{a.CNN} | {a.execucao}' for a in r.itertuples()]
                        ax.barh(labels,r.aux,label='Construção pelos auxiliares',color='#44779c')
                        ax.barh(labels,r.final,left=r.aux,label='Regressor final',color='#d49b56')
                        ax.set_xlabel('Tempo (min)'); ax.set_title(f'K={kval.value} | semente {seed.value}')
                    else:
                        for run,g in r.groupby('execucao'):
                            g=g.sort_values('K'); ax.plot(g.K,g.aux,'o-',label=f'{g.CNN.iloc[0]} | {run}')
                        ax.set_xlabel('K'); ax.set_ylabel('Construção acumulada dos auxiliares (min)')
                    ax.legend(fontsize=7); ax.grid(alpha=.2); fig.tight_layout(); plt.show(); plt.close(fig); display(r)
                    print('Sem extração CNN. As etapas até K são reutilizadas; não somar as curvas para obter o custo total da busca.'); return
                if modo.value=='Ganho temporal por experimento':
                    rows=[]
                    for run,g in t.groupby('execucao'):
                        v=g[g.conjunto=='validacao'].sort_values(['MAE_mm','colunas'])
                        if v.empty or not (v.colunas==0).any(): continue
                        k=int(v.iloc[0].colunas)
                        root=Path(g.pasta.iloc[0])/'temporal'/proto.value/f'seed_{seed.value}'/'unico'
                        p0=pd.read_csv(root/'K_000'/'predicoes.csv'); pk=pd.read_csv(root/f'K_{k:03d}'/'predicoes.csv')
                        p0=p0[p0.conjunto=='teste']; pk=pk[pk.conjunto=='teste']
                        z=p0.merge(pk,on=['indice_dataset','experimento'],suffixes=('_0','_k'),validate='one_to_one')
                        if len(z)!=len(p0) or len(z)!=len(pk) or not np.allclose(z.y_0,z.y_k):
                            raise ValueError('Amostras não pareadas: '+run)
                        for exp,e in z.groupby('experimento'):
                            base=np.mean(abs(e.y_0-e.y_predito_0)); novo=np.mean(abs(e.y_k-e.y_predito_k))
                            rows.append(dict(experimento=exp,config=f'{g.CNN.iloc[0]} | K={k} | {run}',ganho=100*(base-novo)/base if base>0 else np.nan))
                    if not rows: print('Sem pares completos.'); return
                    r=pd.DataFrame(rows).pivot(index='experimento',columns='config',values='ganho')
                    vals=r.to_numpy(); finite=vals[np.isfinite(vals)]; limite=max(1,abs(finite).max()) if len(finite) else 1
                    fig,ax=plt.subplots(figsize=(max(8,len(r.columns)*2),6)); cmap=plt.get_cmap('RdBu').copy(); cmap.set_bad('#dddddd')
                    im=ax.imshow(np.ma.masked_invalid(vals),cmap=cmap,vmin=-limite,vmax=limite,aspect='auto')
                    ax.set_xticks(range(len(r.columns)),r.columns,rotation=30,ha='right',fontsize=8); ax.set_yticks(range(len(r)),r.index)
                    for i in range(len(r)):
                        for j in range(len(r.columns)): ax.text(j,i,f'{vals[i,j]:.1f}%' if np.isfinite(vals[i,j]) else 'N/D',ha='center',va='center',fontsize=8)
                    fig.colorbar(im,ax=ax,label='Redução do MAE de teste (%) — positivo = melhora')
                    ax.set_title(f'K selecionado por MAE global de validação | semente {seed.value}')
                    fig.tight_layout(); plt.show(); plt.close(fig); display(r); return
                r=d[(d.conjunto=='teste')&(d.colunas==kval.value)]
                fig,ax=plt.subplots(figsize=(12,max(4,len(r)*.3))); labels=[]
                for (run,etapa),g in r.groupby(['execucao','etapa']):
                    g=g.set_index('semente')
                    if not {42,43}<=set(g.index): continue
                    i=len(labels); labels.append(f'{g.CNN.iloc[0]} | {etapa} | {run}')
                    ax.plot([g.loc[42,'MAE_mm'],g.loc[43,'MAE_mm']],[i,i],color='gray')
                    ax.scatter(g.loc[42,'MAE_mm'],i,color='#44779c',label='42' if i==0 else None)
                    ax.scatter(g.loc[43,'MAE_mm'],i,color='#d49b56',label='43' if i==0 else None)
                if not labels: plt.close(fig); print('Sem pares completos 42/43 neste K.'); return
                ax.set_yticks(range(len(labels)),labels,fontsize=8); ax.set_xlabel('MAE de teste (mm)'); ax.set_title(f'Variação entre treinamentos | K={kval.value}')
                ax.legend(title='Semente'); ax.grid(axis='x',alpha=.2); fig.tight_layout(); plt.show(); plt.close(fig)
                print('Mesmas partições; duas sementes fornecem apenas uma comparação descritiva. O filtro de semente não se aplica a este gráfico.')
            except (OSError,KeyError,ValueError) as exc:
                print('Não foi possível montar este gráfico:',exc)
    preencher(); grupo.observe(preencher,names='value')
    for w in [grupo,runs,proto,seed,kval,modo]: w.observe(desenhar,names='value')
    display(widgets.VBox([grupo,runs,widgets.HBox([proto,seed,kval]),modo,out]))
    desenhar()
painel_adicional_tcc()
